In [1]:
import re, pathlib, pymupdf, docx

def load_docs(folder):
    docs = []
    for path in sorted(pathlib.Path(folder).glob("*")):
        if path.suffix == ".pdf":
            for i, page in enumerate(pymupdf.open(path)):
                docs.append({"source": path.name, "page": i + 1, "text": page.get_text()})
        elif path.suffix == ".docx":
            text = "\n".join(p.text for p in docx.Document(path).paragraphs)
            docs.append({"source": path.name, "page": None, "text": text})
    return docs

def clean(text):
    text = re.sub(r"\.{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_front_matter(raw_text):
    return len(re.findall(r"\.{5,}", raw_text)) >= 5

def is_useful(text, min_words=25, min_letters=0.35):
    words = text.split()
    if len(words) < min_words:
        return False
    letter_words = re.findall(r"[A-Za-z]{3,}", text)
    return len(letter_words) / len(words) >= min_letters

def chunk_text(text, size=250, overlap=40):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

def make_chunks(docs, size=250, overlap=40):
    chunks, dropped = [], []
    for d in docs:
        if is_front_matter(d["text"]):
            dropped.append({"source": d["source"], "page": d["page"], "text": d["text"][:80]})
            continue
        for c in chunk_text(clean(d["text"]), size, overlap):
            item = {"source": d["source"], "page": d["page"], "text": c}
            (chunks if is_useful(c) else dropped).append(item)
    print(f"kept {len(chunks)}, dropped {len(dropped)}")
    return chunks, dropped

In [2]:
docs = load_docs("papers")
chunks, dropped = make_chunks(docs)

kept 153, dropped 7


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME)

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype="float32")
embeddings.shape

/mnt/c/Users/Justin Ray/Documents/Acads1.0/Machine & Deep Learning/RAG/thesis-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 5/5 [00:17<00:00,  3.57s/it]


(153, 384)

In [4]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
index.ntotal

153

In [ ]:
import ollama

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
LLM = "llama3.2:3b"   

def retrieve(query, k=5):
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [chunks[i] for i in ids[0]]

def ask(question, k=5):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
    messages = [
        {"role": "system", "content": "Answer using only the context. If the answer is not in the context, say you cannot find it. Cite sources like [1]."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    resp = ollama.chat(model=LLM, messages=messages, options={"temperature": 0.2, "num_ctx": 4096})
    print(resp["message"]["content"])
    print("\nSources:")
    for n, h in enumerate(hits, 1):
        print(f"[{n}] {h['source']} p.{h['page']}")

In [11]:
ask("What is the main contribution of CAFRM?")


Based on the provided context, the main contribution of CAFRM is the development of a lightweight convolutional neural network (CNN) called Compression-Aware Forensic Refinement Module (CAFRM) that can predict a residual correction to be added to a degraded input image, rather than generating a new image from scratch. CAFRM uses residual learning, which means that it predicts a residual correction that is added to the degraded input image, rather than generating a new image from scratch. This approach allows CAFRM to capture both fine detail and wider context, and it is designed to work with images that have been compressed using the PDF round-trip process. The main contribution of CAFRM is to improve the performance of tamper localization in document images under PDF compression, by incorporating the detection loss of the ADCD-Net alongside a small-weight pixel loss.

Sources:
[1] CAFRM.pdf p.57
[2] CAFRM.pdf p.72
[3] CAFRM.pdf p.75
[4] CAFRM.pdf p.92
[5] CAFRM.pdf p.25
